# [Querying with SPARQL - RDFLib](https://rdflib.readthedocs.io/en/stable/intro_to_sparql/)

SPARQL: グラフパターンを宣言的に評価して変数束縛を求める言語

In [21]:
from rdflib import Graph, URIRef
from rdflib.namespace import FOAF
from rdflib.plugins.sparql import prepareQuery
from rdflib.plugins.sparql.sparql import Query
from rdflib.query import Result, ResultRow

g: Graph = Graph()

g.parse(
    data="""
        @prefix foaf: <http://xmlns.com/foaf/0.1/> .

        <http://example.org/alice>
            foaf:name "Alice" ;
            foaf:knows <http://example.org/bob> .

        <http://example.org/bob>
            foaf:name "Bob" .

        <http://example.org/carol>
            foaf:name "Carol" ;
            foaf:knows <http://example.org/bob> .
    """,
    format="turtle",
)

print(f"{len(g)} triples")


5 triples


## Run a Query

In [22]:
knows_query: str = """
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT DISTINCT ?aname ?bname
WHERE {
    ?a foaf:knows ?b .
    ?a foaf:name ?aname .
    ?b foaf:name ?bname .
}
"""

qres: Result = g.query(knows_query)

for row in qres:
    assert isinstance(row, ResultRow)
    print(f"{row.aname} knows {row.bname}")

Alice knows Bob
Carol knows Bob


## Update Queries

In [23]:
g_update: Graph = Graph()

g_update.parse(
    data="""
        <x:> a <c:> .
        <y:> a <c:> .
    """,
    format="turtle",
)

qres = g_update.query("""SELECT ?s WHERE { ?s a <c:> }""")

for row in qres:
    assert isinstance(row, ResultRow)
    print(f"{row.s}")

x:
y:


In [24]:
g_update.update("""INSERT DATA { <z:> a <c:> }""")

qres = g_update.query("""SELECT ?s WHERE { ?s a <c:> }""")

print("After update:")

for row in qres:
    assert isinstance(row, ResultRow)
    print(f"{row.s}")

After update:
x:
y:
z:


In [25]:
g_update.update(
    """
    DELETE { <y:> a <c:> }
    INSERT { <y:> a <d:> }
    WHERE  { <y:> a <c:> }
    """
)

print("After second update:")

qres = g_update.query("""SELECT ?s ?o WHERE { ?s a ?o }""")

for row in qres:
    assert isinstance(row, ResultRow)
    print(f"{row.s} a {row.o}")


After second update:
x: a c:
z: a c:
y: a d:


## Querying a Remote Service

In [26]:
import rdflib

remote_g = rdflib.Graph()
qres = remote_g.query(
    """
    SELECT ?s
    WHERE {
      SERVICE <https://dbpedia.org/sparql> {
        ?s a ?o .
      }
    }
    LIMIT 3
    """
)

for row in qres:
    assert isinstance(row, ResultRow)
    print(row.s)


http://www.openlinksw.com/schemas/virtrdf#DefaultQuadMap
http://www.openlinksw.com/schemas/virtrdf#TmpQuadMap
http://www.openlinksw.com/schemas/virtrdf#DefaultServiceMap


## Prepared Queries

In [ ]:
q: Query = prepareQuery(
    "SELECT ?s WHERE { ?person foaf:knows ?s . }",
    initNs={"foaf": FOAF},
)

alice: URIRef = URIRef("http://example.org/alice")

qres = g.query(
    q,
    initBindings={"person": alice}, # prepareしたクエリの ?person に alice をバインドする
)

for row in qres:
    assert isinstance(row, ResultRow)
    print(row)

(rdflib.term.URIRef('http://example.org/bob'),)
